# Perimeter Creation Using HMS Shapefiles

In [1]:
import geopandas as gpd
import os
import pathlib as pl
%matplotlib inline

In [2]:
os.chdir('..')
os.chdir('..')

home = pl.Path(os.getcwd())
print(home)

#location of all larger functions required for the perimeter delineation
from _pyfiles.perimeter_functions import *

C:\code


In [3]:
inputs = home/'inputs'/'wy22_perimeter_updates'
outputs = home/'outputs'/'wy22_perimeter_updates'

if not os.path.exists(inputs):
    os.makedirs(inputs)
if not os.path.exists(outputs):
    os.makedirs(outputs)

#### hide for now ####

## back to normal 

In [ ]:
#reading from shapefiles 
subbasins = gpd.read_file(r"C:\code\inputs\wy22_perimeter_updates\Subbasin.shp")
junctions = gpd.read_file(r"C:\code\inputs\wy22_perimeter_updates\Junction.shp")
sinks = gpd.read_file(r"C:\code\inputs\wy22_perimeter_updates\Sink.shp")

#transforming into dataframes
juncs_df = junctions.to_dataframe()

In [24]:
import geopandas as gpd
from pathlib import Path
import requests

# Load the subbasins
########pending removal 
subbasins = gpd.read_file(r"C:\code\inputs\wy22_perimeter_updates\Subbasin.shp")
########pending removal 

print(f"Successfully loaded shapefile with {len(subbasins)} features")
# Get the bounding box
bbox = subbasins.total_bounds
# Ensure the subbasins are in WGS84 (EPSG:4326) for the web service
if subbasins.crs != 'EPSG:4326':
    print(f"Converting from {subbasins.crs} to EPSG:4326")
    subbasins = subbasins.to_crs('EPSG:4326')
    bbox = subbasins.total_bounds
# Format bbox for the web service
bbox_string = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
# Set up the request
url = "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/5/query"
params = {
    'where': '1=1',
    'outFields': '*',
    'geometry': bbox_string,
    'geometryType': 'esriGeometryEnvelope',
    'spatialRel': 'esriSpatialRelIntersects',
    'f': 'geojson',
    'inSR': '4326',
    'outSR': '4326',
    'returnGeometry': 'true'
}
try:
    # Make the request
    response = requests.get(url, params=params)  
    
    if response.status_code == 200:
        # Save the GeoJSON to a temporary file
        temp_geojson = "temp_hucs.geojson"
        with open(temp_geojson, 'w') as f:
            f.write(response.text)
        # Read the temporary file
        hucs = gpd.read_file(temp_geojson)
        print(f"Successfully loaded {len(hucs)} HUC features")
        hucs.to_file(outputs/"huc10s_to_compare.shp")
        
except Exception as e:
    print(f"An error occurred: {str(e)}")
    print(f"Error type: {type(e)}")

Successfully loaded shapefile with 932 features
Converting from PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["US survey foot",0.304800609601219],AXIS["Easting",EAST],AXIS["Northing",NORTH]] to EPSG:4326
Successfully loaded 40 HUC features


C:\Users\magomez\AppData\Local\Temp\ipykernel_31524\2821087516.py:41: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  hucs.to_file(outputs/"huc10s_to_compare.shp")


In [ ]:
#WBD HUCs that are around the data we have available to be converted into a dataframe
hucs_df = hucs.to_dataframe()

#spatial join junctions and huc10s from wbd to assign each junction
joined_gdf = gpd.sjoin(juncs_df, hucs_df, how='left', op='intersects')
joined_gdf['huc'] = gdf2['']


In [1]:
#preparing junction dataframe to fill with information 
junc_df['HUC10_mod'] = None

NameError: name 'junc_df' is not defined

In [ ]:
junc_df

In [ ]:
junc_df.plot()

In [ ]:
#requires clean up and test

#dataframe 1 in which there is the points from the junctions 

